# Lab 8: Introduction au Framework ADK et Multi-Provider

**Navigation** : [Index](../../README.md) | [Précédent <<](../../Track1-LangChain/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >>](Lab9-First-ADK-Agent.ipynb)

## Objectifs d'apprentissage

A la fin de ce laboratoire, vous saurez :
1. Expliquer l'architecture du Google Agent Development Kit (ADK)
2. Configurer un environnement multi-provider (Gemini, vLLM, OpenAI)
3. Créer un premier client LLM avec votre provider choisi
4. Comparer les réponses de différents providers sur le même prompt

### Prérequis
- Python 3.10+
- Fichier `.env` configuré avec au moins un provider (voir section Configuration)
- Connaissance de base des LLMs (API, tokens, temperature)

### Durée estimée : 45-60 minutes

***

## Configuration requise

Avant de commencer, assurez-vous d'avoir configuré votre fichier `.env` :

```bash
# Exemple de configuration .env
ACTIVE_PROVIDER=gemini  # ou vllm, openai

# Gemini (optionnel)
GOOGLE_API_KEY=your_gemini_key

# vLLM (optionnel)
VLLM_BASE_URL=https://your-vllm-endpoint.com/v1
VLLM_MODEL=Qwen/Qwen2.5-72B-Instruct

# OpenAI (optionnel)
OPENAI_API_KEY=sk-...
```

***

## 1. Architecture Google ADK

Le **Agent Development Kit (ADK)** de Google est un framework pour construire des agents IA avec :

> **Repères bibliographiques.** Le concept d'agent IA à base de LLM (LLM-as-agent : perception → raisonnement → action via *tools*) est formalisé dans la synthèse de référence Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Le framework **Google ADK** (Agent Development Kit) en fournit une implémentation officielle (`google.github.io/adk-docs`, dépôt `google/adk-python`), et **LangChain** — lancé en octobre 2022 par H. Chase — popularise l'approche modulaire (composabilité de chaînes, *tools*, mémoire) sur laquelle s'appuie une large part de l'écosystème agent Python.

- **Agents** : Entités qui interagissent via des tools
- **Tools** : Fonctions que l'agent peut appeler
- **Sessions** : Gestion de l'état conversationnel
- **Memory** : Persistance du contexte

### Comparaison avec LangChain

| Aspect | ADK | LangChain |
|--------|-----|----------|
| **Philosophie** | Google-first, intégré GCP | Multi-provider natif |
| **Agents** | Agent classes avec tools | Runnable, chains, agents |
| **Mémoire** | Session state intégrée | Modules séparés |
| **Déploiement** | Vertex AI Agent Engine | Variable |
| **Providers** | Gemini natif, OpenAI compatible | 50+ providers |

## 2. Configuration de l'Environnement

### Installation des dépendances

In [1]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install -r ../requirements.txt

print("Dependances : cellule d'installation (decommenter si necessaire)")

Dependances : cellule d'installation (decommenter si necessaire)


Import des modules et verification de l'installation.

In [2]:
import sys
from pathlib import Path

# Add parent directory (Track2-GoogleADK) to path for config/utils imports
sys.path.insert(0, str(Path().resolve().parent))

from config import get_settings, get_provider_config, ProviderType
from utils import LLMClient, generate
import os

print("Imports OK : config (get_settings, get_provider_config, ProviderType), utils (LLMClient, generate)")

Imports OK : config (get_settings, get_provider_config, ProviderType), utils (LLMClient, generate)


### Vérification de la configuration

In [3]:
# Chargement des settings
settings = get_settings()

print(f"Provider actif: {settings.active_provider}")
print(f"Modèle configuré: {settings.gemini_model if settings.active_provider == 'gemini' else 'autre'}")

Provider actif: openai
Modèle configuré: autre


Configuration detaillee du provider LLM.

In [4]:
# Configuration détaillée du provider
config = get_provider_config(settings)
print(f"Provider: {config.provider.value}")
print(f"Modèle: {config.model}")
print(f"Base URL: {config.base_url or 'défaut'}")
print(f"API Key configurée: {'Oui' if config.api_key else 'Non'}")

Provider: openai
Modèle: gpt-4o
Base URL: https://api.openai.com/v1
API Key configurée: Oui


## 3. Premier Test avec le Client LLM

Utilisons notre couche d'abstraction pour envoyer un prompt simple.

In [5]:
# Création du client
client = LLMClient()
print(client)

LLMClient(provider=openai, model=gpt-4o)


Test de generation simple avec le client configure.

In [6]:
# Test de génération simple
response = client.generate(
    "Explique ce qu'est un agent IA en 2 phrases.",
    temperature=0.7
)
print(response)

Un agent IA est un programme informatique conçu pour percevoir son environnement, prendre des décisions autonomes et effectuer des actions afin d'atteindre des objectifs spécifiques. Il utilise des algorithmes d'intelligence artificielle pour analyser des données, apprendre de ses expériences et s'adapter aux changements dans son environnement.


### Test avec prompt système

In [7]:
response = client.generate(
    prompt="Quel est le meilleur algorithme pour classifier des données tabulaires?",
    system="Tu es un expert en machine learning. Réponds de façon concise et technique.",
    temperature=0.3
)
print(response)

Il n'existe pas de "meilleur" algorithme universel pour classifier des données tabulaires, car la performance dépend des caractéristiques spécifiques du jeu de données. Cependant, quelques algorithmes performants pour les données tabulaires incluent :

1. **Random Forest** : Robuste et souvent performant sur des données tabulaires avec des interactions complexes.
2. **Gradient Boosting Machines (GBM)** : Comme XGBoost, LightGBM ou CatBoost, qui sont efficaces pour capturer des relations non linéaires.
3. **Logistic Regression** : Simple et efficace pour des problèmes linéaires ou lorsque l'interprétabilité est importante.
4. **Support Vector Machines (SVM)** : Efficace pour des jeux de données avec des marges claires entre les classes.

Il est souvent recommandé de tester plusieurs algorithmes et d'utiliser la validation croisée pour déterminer celui qui fonctionne le mieux pour votre cas spécifique.


## 4. Comparaison Multi-Provider

Testons le même prompt avec différents providers pour comparer les réponses.

In [8]:
test_prompt = "Donne 3 bonnes pratiques pour nettoyer un dataset."
test_system = "Réponds en français, de façon structurée avec des bullet points."

# Test avec le provider actuel
print(f"=== Provider: {settings.active_provider.upper()} ===")
response = client.generate(test_prompt, system=test_system)
print(response)

=== Provider: OPENAI ===


Voici trois bonnes pratiques pour nettoyer un dataset :

- **Traitement des valeurs manquantes :**
  - Identifier les valeurs manquantes dans le dataset.
  - Décider comment les gérer : soit en les supprimant, soit en les remplaçant par une estimation (comme la moyenne, la médiane ou une valeur par défaut appropriée).
  - Utiliser des techniques comme l'imputation ou les modèles prédictifs pour combler les lacunes lorsque les données manquantes sont significatives.

- **Gestion des doublons :**
  - Rechercher et identifier les doublons dans le dataset qui peuvent biaiser les analyses.
  - Supprimer les doublons pour garantir que chaque observation est unique, à moins que les doublons ne soient justifiés par la nature des données.
  - Vérifier que la suppression des doublons ne supprime pas des informations importantes.

- **Standardisation et correction des données :**
  - Vérifier l'uniformité des formats d'entrée, tels que les dates, les unités de mesure, et les représentations textu

### Interprétation

La réponse ci-dessus provient du provider configuré dans `.env` (`ACTIVE_PROVIDER`).

Pour tester un autre provider, modifiez votre fichier `.env` et redémarrez le kernel, ou instanciez un client avec une configuration explicite :

```python
from config import ProviderConfig, ProviderType

# Exemple pour vLLM
vllm_config = ProviderConfig(
    provider=ProviderType.VLLM,
    model="Qwen/Qwen2.5-72B-Instruct",
    base_url="https://your-vllm-endpoint.com/v1",
    api_key=None
)
vllm_client = LLMClient(vllm_config)
```

## 5. Interface de Chat avec Historique

Le client supporte également une interface de chat avec historique des messages.

In [9]:
# Conversation multi-tours
messages = [
    {"role": "user", "content": "Je veux analyser un dataset de ventes."},
]

response1 = client.chat(messages)
print("Assistant:", response1)

# Ajout de la réponse et continuation
messages.append({"role": "assistant", "content": response1})
messages.append({"role": "user", "content": "Quelles visualisations me recommandes-tu?"})

response2 = client.chat(messages)
print("\nAssistant:", response2)

Assistant: Pour analyser un dataset de ventes, vous pouvez suivre plusieurs étapes clés afin de tirer des conclusions utiles et de découvrir des tendances intéressantes. Voici un guide général sur la façon de procéder :

1. **Compréhension des données** :
   - **Chargement des données** : Utilisez des outils comme Excel, Google Sheets, Python (avec pandas), ou R pour charger vos données.
   - **Exploration initiale** : Examinez les premières lignes du dataset pour comprendre sa structure et les types de données qu'il contient.
   - **Description des données** : Utilisez des fonctions descriptives pour obtenir des statistiques sommaires (moyenne, médiane, minimum, maximum, écart-type).

2. **Nettoyage des données** :
   - **Gestion des valeurs manquantes** : Identifiez et traitez les valeurs manquantes via l'imputation ou la suppression.
   - **Correction des erreurs** : Cherchez des incohérences ou des erreurs de saisie et corrigez-les.
   - **Normalisation des formats** : Assurez-vous


Assistant: Le choix des visualisations dépend des questions spécifiques que vous souhaitez explorer dans votre dataset de ventes. Voici quelques visualisations courantes et utiles pour analyser des données de ventes :

1. **Graphiques à barres** :
   - **Comparaison des ventes par catégories** : Utilisez des graphiques à barres pour comparer les ventes entre différents produits, catégories, ou régions.
   - **Évolution des ventes par période** : Un graphique à barres empilées peut montrer la contribution de chaque catégorie aux ventes totales sur une période donnée.

2. **Graphiques en ligne (line charts)** :
   - **Tendances temporelles** : Idéal pour montrer l'évolution des ventes au fil du temps (par jour, semaine, mois, trimestre, ou année).
   - **Saisonnalité** : Identifiez les schémas saisonniers dans les données.

3. **Graphiques à secteurs (pie charts)** :
   - **Répartition des ventes** : Visualisez la part de marché ou la répartition des ventes entre différentes catégories 

## 6. Architecture des Frameworks DS-STAR et MLE-STAR

DS-STAR et MLE-STAR sont deux agents de référence (State-of-the-Art) conçus par Google Research pour la data science et l'ingénierie ML — ils incarnent le paradigme d'agent LLM *planner-coder* (décomposition de tâche → génération de code → exécution → raffinement) décrit par Xi et al. (2025) et appliqué à la compétition Kaggle et au cycle d'expérimentation ML.

Ce track Track2-GoogleADK intègre les frameworks de recherche Google :

### DS-STAR (Data Science Agent)

Architecture Planner-Coder-Verifier pour la data science autonome :

```mermaid
flowchart TD
    FA["File Analyzer"] --> P["Planner"]
    P --> C["Coder"]
    P --> V["Verifier"]
    C --> E["Executor"]
    E --> V
```

Le planificateur distribue le travail entre génération et vérification, tandis que l'exécuteur renvoie les résultats au vérificateur pour fermer la boucle de raffinement.

**Performance** : 45.2% accuracy sur DABStep benchmark

### MLE-STAR (ML Engineering Agent)

Extension avec recherche web et optimisation automatique :

- Web Search pour modèles SOTA
- Ablation studies ciblées
- Ensemble stratégies automatisées

**Performance** : 63.6% médailles sur MLE-Bench-Lite

## Exercice : Comparaison Multi-Provider

Maintenant que vous avez compris l'architecture, testez votre capacité à utiliser différents providers pour la même tâche.

In [10]:
# Exercice : Comparaison Multi-Provider
# Objectif : Créer une fonction qui compare les réponses de 3 providers sur le même prompt

# TODO: Définissez un prompt de test pertinent pour la data science
test_prompt = None  # Exemple: "Quelles sont les 3 étapes clés pour préparer un dataset pour le ML?"
test_system = None  # Exemple: "Réponds en français de façon concise."

# TODO: Créez une fonction compare_providers qui teste le prompt sur plusieurs providers
def compare_providers(prompt, system_prompt, providers=None):
    """
    Compare les réponses de différents providers pour un prompt donné.
    
    Args:
        prompt: Le prompt utilisateur
        system_prompt: Le prompt système
        providers: Liste de providers à tester (ex: ['gemini', 'vllm'])
    
    Returns:
        dict: Réponses par provider avec temps de réponse
    """
    # Indice: Utilisez ProviderConfig pour configurer chaque provider
    # Indice: Mesurez le temps avec time.time()
    results = {}
    
    # TODO: Implémentez la boucle de test pour chaque provider
    
    return results

# TODO: Exécutez la comparaison et affichez les résultats
# results = compare_providers(test_prompt, test_system)
# for provider, data in results.items():
#     print(f"\n=== {provider.upper()} ({data['time']:.2f}s) ===")
#     print(data['response'])

print("Exercice à compléter - voir les TODO ci-dessus")

Exercice à compléter - voir les TODO ci-dessus


## Exercice : Chat Multi-Tours avec Contexte Data Science

Utilisez l'interface `chat()` du client LLM pour construire une conversation specialisee en data science. L'objectif est de simuler un assistant qui guide un utilisateur dans son analyse de données étapes par étapes.

### Objectifs
1. Construire un historique de conversation avec 3 echanges
2. Utiliser un system prompt specialise data science
3. Observer comment le contexte précédent influence les reponses

**Indice :**
- Utilisez `client.chat(messages)` avec une liste de dictionnaires `{"rôle": "user"/"assistant", "content": "..."}`
- Commencez par un message system via le premier élément de la liste

In [11]:
# Exercice : Chat Multi-Tours specialise Data Science
# Objectif : Simuler un assistant d'analyse de donnees en 3 echanges

# TODO: Initialisez la liste de messages avec un message system
messages = None  # Remplacez par [{"role": "system", "content": "Tu es un expert en data science..."}]

# TODO: Premier echange - l'utilisateur decrit son dataset
# messages.append({"role": "user", "content": "J'ai un dataset de ventes avec date, produit, region, quantite et prix."})
# response_1 = client.chat(messages)
# messages.append({"role": "assistant", "content": response_1})

# TODO: Deuxieme echange - l'utilisateur demande une analyse specifique
# messages.append({"role": "user", "content": "Quelles visualisations me recommandes-tu pour comparer les regions ?"})
# response_2 = client.chat(messages)
# messages.append({"role": "assistant", "content": response_2})

# TODO: Troisieme echange - approfondissement
# messages.append({"role": "user", "content": "Comment detecter des valeurs aberrantes dans les prix ?"})
# response_3 = client.chat(messages)

# TODO: Affichez les 3 reponses et observez la progression du contexte
# print("=== Echange 1 ===")
# print(response_1)
# print("\n=== Echange 2 ===")
# print(response_2)
# print("\n=== Echange 3 ===")
# print(response_3)

print("Exercice a completer : chat multi-tours avec contexte data science")

Exercice a completer : chat multi-tours avec contexte data science


## Exercice : Exploration des Paramètres de Generation

Experimentez avec les paramètres `temperature` et `max_tokens` pour comprendre leur impact sur la qualite des reponses d'un agent. L'objectif est de trouver les paramètres optimaux pour différentes tâches d'agent.

### Objectifs
1. Tester 3 valeurs de temperature (0.1, 0.7, 1.5) sur un prompt technique
2. Observer l'impact de `max_tokens` sur la longueur des reponses
3. Determiner les paramètres ideaux pour du code generation vs. du texte creatif

**Indice :**
- `client.generate(prompt, temperature=0.1, max_tokens=100)` pour contrôler la generation
- Temperature basse = reponses déterministes (ideal pour du code)
- Temperature haute = reponses creatives (ideal pour du brainstorming)

In [12]:
# Exercice : Exploration des parametres temperature et max_tokens
# Objectif : Trouver les parametres optimaux selon la tache de l'agent

prompt_technique = "Explique la difference entre un agent base sur des tools et un agent base sur du code execution."

# TODO: Testez 3 temperatures et comparez les reponses
temperatures = [0.1, 0.7, 1.5]
results = {}

# Indice: bouclez sur temperatures et appelez client.generate(prompt_technique, temperature=t)
# for t in temperatures:
#     results[t] = client.generate(prompt_technique, temperature=t)
#     print(f"--- Temperature {t} ---")
#     print(results[t][:200])
#     print()

# TODO: Testez l'impact de max_tokens sur un prompt de code
prompt_code = "Ecris une fonction Python qui calcule la moyenne mobile d'une serie temporelle."

# Indice: comparez max_tokens=50 vs max_tokens=500
# resp_short = client.generate(prompt_code, temperature=0.2, max_tokens=50)
# resp_long = client.generate(prompt_code, temperature=0.2, max_tokens=500)
# print(f"Short (50 tokens): {resp_short[:100]}...")
# print(f"Long (500 tokens): {resp_long[:200]}...")

# TODO: Resumez vos conclusions dans un dictionnaire
parametres_ideaux = {
    "code_generation": {"temperature": None, "max_tokens": None},  # Remplacez None
    "analyse_creative": {"temperature": None, "max_tokens": None},  # Remplacez None
    "brainstorming": {"temperature": None, "max_tokens": None}   # Remplacez None
}

print("Exercice a completer : exploration des parametres temperature et max_tokens")

Exercice a completer : exploration des parametres temperature et max_tokens


## Résumé et Prochaines Étapes

### Ce que nous avons appris

1. **Configuration multi-provider** : Un seul fichier `.env` permet de switcher entre Gemini, vLLM, OpenAI
2. **Abstraction LiteLLM** : Interface unifiée pour tous les providers
3. **Client LLM simple** : `generate()` et `chat()` pour interagir avec n'importe quel modèle
4. **Architecture DS-STAR** : Framework Planner-Coder-Verifier pour la data science autonome

### Points clés à retenir

| Concept | Description |
|---------|-------------|
| `ProviderConfig` | Configuration d'un provider LLM |
| `LLMClient` | Client unifié pour tous les providers |
| `generate()` | Génération simple avec prompt |
| `chat()` | Conversation multi-tours avec historique |

### Prochaines étapes

- **Lab 9** : Créer un premier agent ADK avec tools Python pour analyser des DataFrames
- **Lab 10** : Implémenter le File Analyzer de DS-STAR
- **Lab 11** : Boucle Planner-Coder-Verifier

***

**Navigation** : [Index](../../README.md) | [Précédent <<](../../Track1-LangChain/Day3-Data-Agents/Labs/Lab7-Data-Analysis-Agent/Lab7-Data-Analysis-Agent.ipynb) | [Suivant >> Lab 9 - First ADK Agent](Lab9-First-ADK-Agent.ipynb)

## Ressources

- [Google ADK Documentation](https://github.com/google/adk-samples)
- [DS-STAR Paper](https://research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/)
- [MLE-STAR Paper](https://research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/)
- [LiteLLM Documentation](https://docs.litellm.ai/)

## Références

1. Z. Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023. Synthèse de référence sur les agents IA à base de LLM (architecture perception-raisonnement-action, *tools*, mémoire, cadres planner-coder).
2. Google, *Agent Development Kit (ADK)*, documentation officielle, `google.github.io/adk-docs` (dépôt `google/adk-python`). Framework agent-first de Google (sessions, mémoire, Vertex AI Agent Engine).
3. H. Chase, *LangChain*, octobre 2022, `langchain.com` / `github.com/langchain-ai/langchain`. Framework modulaire open-source pour applications LLM (chaînes composables, *tools*, mémoire) — référence de l'écosystème agent Python.
4. Google Research, *DS-STAR: A State-of-the-Art Versatile Data Science Agent*, 2025, `research.google/blog/ds-star-a-state-of-the-art-versatile-data-science-agent/`. Agent SOTA data science (paradigme planner-coder).
5. Google Research, *MLE-STAR: A State-of-the-Art Machine Learning Engineering Agent*, 2025, `research.google/blog/mle-star-a-state-of-the-art-machine-learning-engineering-agents/`. Agent SOTA ingénierie ML (cycle d'expérimentation, Kaggle).